# Advanced Tutorial Problems: `yield from`, Closing, Exceptions, and Return Values

This is a **new problem set** on generator delegation.

The notebook follows a tutorial style:

- each problem begins with a concrete question,
- the problem is broken into small logical steps,
- you are asked to predict behavior before running code,
- observations are checked with assertions,
- and every problem ends with a complete explanation and solution.

The central topic is the protocol created by:

- `yield from`
- `next()`
- `send()`
- `throw()`
- `close()`
- `GeneratorExit`
- generator `return` values

## How to use this notebook

For every problem:

1. Read the scenario.
2. Pause at the **Prediction** cell.
3. Write down what you think will happen.
4. Run the experiment one cell at a time.
5. Compare your prediction with the observed generator states and event logs.
6. Read the worked explanation.
7. Modify the extension example.

The code uses assertions so that subtle protocol mistakes are detected immediately.

## What makes these problems advanced?

The difficult part of generators is not yielding a sequence of values.

The difficult part is understanding that a generator may participate in a **bidirectional protocol**:

- the caller can request another value,
- send a value inward,
- inject an exception,
- request cancellation,
- and receive a final return value.

When `yield from` is added, that protocol crosses one or more delegation boundaries.

## Compatibility note

Python 3.13 changed one detail:

When a generator catches `GeneratorExit` and returns a value, `generator.close()` can return that value on Python 3.13 and later. Older Python versions discard it.

The notebook detects the version whenever that distinction matters.

# Shared tools

We will use a structured event log instead of relying only on `print()`.

This gives us two advantages:

- event order can be tested,
- and values associated with events are preserved.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from inspect import getgeneratorlocals, getgeneratorstate
from typing import Any, Iterable
import sys

print("Python version:", sys.version.split()[0])

Python version: 3.13.7


In [2]:
@dataclass
class EventLog:
    events: list[tuple[str, Any]] = field(default_factory=list)

    def add(self, name: str, value: Any = None) -> None:
        self.events.append((name, value))

    def names(self) -> list[str]:
        return [name for name, _ in self.events]

    def show(self) -> None:
        for index, (name, value) in enumerate(self.events, start=1):
            print(f"{index:02d}. {name}: {value!r}")


def collect_with_return(generator):
    yielded = []

    while True:
        try:
            yielded.append(next(generator))
        except StopIteration as exc:
            return yielded, exc.value


def assert_state(generator, expected: str) -> None:
    actual = getgeneratorstate(generator)
    assert actual == expected, f"expected {expected}, got {actual}"


def close_returns_values() -> bool:
    return sys.version_info >= (3, 13)

The helper `collect_with_return()` is important.

Calling `list(generator)` collects yielded values, but it discards the final `StopIteration.value`.

Our helper preserves both:

```python
yielded_values, final_return_value = collect_with_return(generator)
```

# Problem 1 — A return value crossing two delegation boundaries

We will build three generators:

```text
outer
  └── middle
        └── leaf
```

The `leaf` generator yields two values and then returns a number.

The `middle` generator captures that number, transforms it, and returns another number.

The `outer` generator captures the middle result and yields a final report.

## Step 1 — Define the leaf generator

Before running it, answer:

- Which values are yielded?
- Which value is returned?
- Is the returned value part of the yielded sequence?

In [3]:
def leaf(log: EventLog):
    log.add("leaf-start")
    yield 4
    yield 7
    log.add("leaf-return")
    return 11

## Step 2 — Define the middle generator

Notice that `yield from leaf(log)` is an **expression**.

Its value will be assigned to `leaf_result`.

In [4]:
def middle(log: EventLog):
    log.add("middle-start")
    leaf_result = yield from leaf(log)
    log.add("middle-received", leaf_result)
    return leaf_result * 3

## Step 3 — Define the outer generator

The outer generator delegates to the middle generator.

After the middle generator finishes, the outer generator yields one report.

In [5]:
def outer(log: EventLog):
    log.add("outer-start")
    middle_result = yield from middle(log)
    log.add("outer-received", middle_result)
    yield {"final": middle_result}
    log.add("outer-end")

## Prediction

Before running the next cell, predict:

1. The complete yielded sequence.
2. The final return value of `outer`.
3. The order of all log events.

Remember: values returned by subgenerators are not automatically yielded.

In [6]:
log = EventLog()
yielded, returned = collect_with_return(outer(log))

print("Yielded:", yielded)
print("Returned:", returned)
log.show()

Yielded: [4, 7, {'final': 33}]
Returned: None
01. outer-start: None
02. middle-start: None
03. leaf-start: None
04. leaf-return: None
05. middle-received: 11
06. outer-received: 33
07. outer-end: None


## Solution

The leaf yields `4` and `7`, then returns `11`.

The middle generator receives `11` as the value of its `yield from` expression. It returns `33`.

The outer generator receives `33` and yields `{"final": 33}`.

The complete yielded sequence is therefore:

```python
[4, 7, {"final": 33}]
```

The outer generator has no explicit `return`, so its final return value is `None`.

In [7]:
assert yielded == [4, 7, {"final": 33}]
assert returned is None

assert log.events == [
    ("outer-start", None),
    ("middle-start", None),
    ("leaf-start", None),
    ("leaf-return", None),
    ("middle-received", 11),
    ("outer-received", 33),
    ("outer-end", None),
]

## Extension

Change `leaf` so that it returns the sum of all values it yielded.

Then change `middle` so that it returns both the leaf result and the number of yielded values.

# Problem 2 — Sending values through nested delegation

Now we will study `send()`.

A caller sends values to the outer generator, but the active receiver is a nested subgenerator.

The delegation chain is:

```text
caller → session → collector
```

## Step 1 — Define a sentinel

A sentinel gives the protocol a clear normal-completion message.

This is usually better than using `close()` to mean “finish successfully and calculate a report.”

In [8]:
FINISH = object()

## Step 2 — Define the collector

Protocol:

1. The first `next()` yields `"collector-ready"`.
2. Each sent value is appended to a list.
3. The collector yields the current number of collected items.
4. Sending `FINISH` terminates normally.
5. The collector returns a summary dictionary.

In [9]:
def collector(log: EventLog):
    values = []
    log.add("collector-start")

    incoming = yield "collector-ready"

    while incoming is not FINISH:
        log.add("collector-received", incoming)
        values.append(incoming)
        incoming = yield len(values)

    log.add("collector-finish")
    return {
        "count": len(values),
        "values": tuple(values),
    }

## Step 3 — Define the delegating session

The session does not manually forward values.

`yield from` performs the forwarding.

In [10]:
def collection_session(log: EventLog):
    log.add("session-start")
    report = yield from collector(log)
    log.add("session-report", report)
    yield ("report", report)
    log.add("session-end")

## Step 4 — Prime the session

Calling the generator function does not execute its body.

The first `next()` starts the outer generator and then the collector.

In [11]:
log = EventLog()
session = collection_session(log)

assert_state(session, "GEN_CREATED")
first = next(session)

print("First yielded value:", first)
print("Session state:", getgeneratorstate(session))
print("Active subgenerator:", session.gi_yieldfrom)

First yielded value: collector-ready
Session state: GEN_SUSPENDED
Active subgenerator: <generator object collector at 0x000001D82F0AF300>


At this point:

- the session is suspended at `yield from`,
- the collector is suspended at its first `yield`,
- and `session.gi_yieldfrom` points to the collector.

In [12]:
subgenerator = session.gi_yieldfrom

assert first == "collector-ready"
assert subgenerator is not None
assert_state(session, "GEN_SUSPENDED")
assert_state(subgenerator, "GEN_SUSPENDED")

## Step 5 — Send values to the outer generator

Predict which generator receives each value.

In [13]:
print(session.send("alpha"))
print(session.send("beta"))
print(session.send("gamma"))

1
2
3


The caller invoked `session.send(...)`.

However, while `yield from` is active, non-`None` values are forwarded to the collector's `send()` method.

In [14]:
assert log.events[:5] == [
    ("session-start", None),
    ("collector-start", None),
    ("collector-received", "alpha"),
    ("collector-received", "beta"),
    ("collector-received", "gamma"),
]

## Step 6 — Finish normally

Sending the sentinel causes the collector to return.

That return value becomes the value of the session's `yield from` expression.

In [15]:
final_yield = session.send(FINISH)

print(final_yield)
log.show()

('report', {'count': 3, 'values': ('alpha', 'beta', 'gamma')})
01. session-start: None
02. collector-start: None
03. collector-received: 'alpha'
04. collector-received: 'beta'
05. collector-received: 'gamma'
06. collector-finish: None
07. session-report: {'count': 3, 'values': ('alpha', 'beta', 'gamma')}


In [16]:
assert final_yield == (
    "report",
    {"count": 3, "values": ("alpha", "beta", "gamma")},
)

assert next(session, None) is None
assert_state(session, "GEN_CLOSED")

## Main lesson

During active delegation:

```python
outer.send(value)
```

behaves conceptually like:

```python
active_subgenerator.send(value)
```

The delegator becomes active again only when the subgenerator yields, raises an exception, or terminates.

# Problem 3 — Closing the child is not the same as closing the parent

This problem compares two experiments:

- Experiment A: close only the active subgenerator.
- Experiment B: close the delegator.

The difference is subtle and important.

## Step 1 — Define the generators

In [17]:
def child(log: EventLog):
    log.add("child-start")

    try:
        while True:
            value = yield "child-waiting"
            log.add("child-value", value)
    finally:
        log.add("child-finally")


def parent(log: EventLog):
    log.add("parent-start")

    try:
        yield from child(log)
        log.add("parent-after-yield-from")
        yield "parent-resumed"
    finally:
        log.add("parent-finally")

## Experiment A — Close only the child

We obtain the child through `parent_generator.gi_yieldfrom`.

In [18]:
log_a = EventLog()
parent_a = parent(log_a)

assert next(parent_a) == "child-waiting"

child_a = parent_a.gi_yieldfrom
assert child_a is not None

child_a.close()

## Prediction

Immediately after `child_a.close()`:

- Is the child closed?
- Is the parent closed?
- Has the parent already executed the code after `yield from`?

In [19]:
print("Child state:", getgeneratorstate(child_a))
print("Parent state:", getgeneratorstate(parent_a))
log_a.show()

Child state: GEN_CLOSED
Parent state: GEN_SUSPENDED
01. parent-start: None
02. child-start: None
03. child-finally: None


## Explanation

The child closes immediately.

The parent remains suspended. Closing the child does not automatically execute the suspended parent frame.

The parent notices the child's termination only when the caller resumes the parent.

In [20]:
assert_state(child_a, "GEN_CLOSED")
assert_state(parent_a, "GEN_SUSPENDED")
assert log_a.events == [("parent-start", None), ("child-start", None), ("child-finally", None)]

## Resume the parent

The next `next(parent_a)` resumes execution after `yield from`.

In [21]:
resumed_value = next(parent_a)

print("Resumed value:", resumed_value)
log_a.show()

Resumed value: parent-resumed
01. parent-start: None
02. child-start: None
03. child-finally: None
04. parent-after-yield-from: None


In [22]:
assert resumed_value == "parent-resumed"
assert "parent-after-yield-from" in log_a.names()

assert next(parent_a, None) is None
assert_state(parent_a, "GEN_CLOSED")

## Experiment B — Close the parent

Now create a fresh chain and close the delegator instead.

In [23]:
log_b = EventLog()
parent_b = parent(log_b)

assert next(parent_b) == "child-waiting"
child_b = parent_b.gi_yieldfrom

parent_b.close()

Closing the delegator propagates closure inward to the active subgenerator.

After the child finishes cleaning up, the parent also cleans up.

In [24]:
print("Child state:", getgeneratorstate(child_b))
print("Parent state:", getgeneratorstate(parent_b))
log_b.show()

Child state: GEN_CLOSED
Parent state: GEN_CLOSED
01. parent-start: None
02. child-start: None
03. child-finally: None
04. parent-finally: None


In [25]:
assert_state(child_b, "GEN_CLOSED")
assert_state(parent_b, "GEN_CLOSED")

assert log_b.names() == [
    "parent-start",
    "child-start",
    "child-finally",
    "parent-finally",
]

## Conclusion

Direct child close:

```text
child closes → parent remains suspended → caller must resume parent
```

Parent close:

```text
close propagates inward → child cleans up → parent cleans up → whole chain closes
```

# Problem 4 — Exception routing and selective recovery

A caller can inject an exception with `throw()`.

While `yield from` is active, the exception is normally routed to the active subgenerator.

We will handle one exception locally and transform another exception.

## Step 1 — Define the worker

The worker will:

- recover from `ValueError`,
- transform `LookupError` into `RuntimeError`,
- allow unrelated exceptions to escape.

In [26]:
def exception_worker(log: EventLog):
    while True:
        try:
            command = yield "worker-ready"
        except ValueError as exc:
            log.add("value-error", str(exc))
            yield ("recovered", str(exc))
        except LookupError as exc:
            log.add("lookup-error", str(exc))
            raise RuntimeError("lookup operation failed") from exc
        else:
            log.add("command", command)

## Step 2 — Define the parent

The parent's `finally` block proves that cleanup occurs when an unhandled exception terminates delegation.

In [27]:
def exception_parent(log: EventLog):
    log.add("parent-enter")

    try:
        yield from exception_worker(log)
    finally:
        log.add("parent-cleanup")

## Step 3 — Recover from `ValueError`

In [28]:
log = EventLog()
g = exception_parent(log)

assert next(g) == "worker-ready"

recovery = g.throw(ValueError("invalid command"))
print(recovery)

('recovered', 'invalid command')


In [29]:
assert recovery == ("recovered", "invalid command")
assert_state(g, "GEN_SUSPENDED")

assert next(g) == "worker-ready"
assert g.send("continue") == "worker-ready"

The worker caught the injected exception and remained alive.

The delegation chain is still suspended and usable.

## Step 4 — Transform `LookupError`

Now inject a `KeyError`, which is a subclass of `LookupError`.

In [30]:
try:
    g.throw(KeyError("missing"))
except RuntimeError as exc:
    transformed_exception = exc
    print(type(exc).__name__, str(exc))
else:
    raise AssertionError("Expected RuntimeError")

RuntimeError lookup operation failed


The worker transformed the exception.

Because the transformed exception was not handled by the parent, it escaped to the caller and closed the parent.

In [31]:
assert str(transformed_exception) == "lookup operation failed"
assert isinstance(transformed_exception.__cause__, KeyError)
assert_state(g, "GEN_CLOSED")

assert log.names() == [
    "parent-enter",
    "value-error",
    "command",
    "lookup-error",
    "parent-cleanup",
]

## Best practice

Catch only exceptions that belong to the generator's documented protocol.

Broad exception handling can hide defects and leave a long-running generator in an invalid state.

# Problem 5 — Why yielding during close is illegal

Calling `close()` injects `GeneratorExit`.

A generator is allowed to:

- clean up,
- return,
- or re-raise `GeneratorExit`.

It is not allowed to yield another value.

## Step 1 — Create a broken generator

In [32]:
def broken_on_close():
    try:
        yield "running"
    except GeneratorExit:
        yield "this yield is illegal"

## Prediction

What should happen when `close()` reaches the second `yield`?

In [33]:
broken = broken_on_close()
assert next(broken) == "running"

try:
    broken.close()
except RuntimeError as exc:
    print(type(exc).__name__, str(exc))
    close_error = exc
else:
    raise AssertionError("Expected RuntimeError")

RuntimeError generator ignored GeneratorExit


In [34]:
assert "ignored GeneratorExit" in str(close_error)

## Explanation

`close()` means:

> Stop producing values and terminate.

If the generator yields after receiving `GeneratorExit`, it violates that contract. Python therefore raises `RuntimeError`.

## Step 2 — Write the correct pattern

Use `finally` for unconditional resource cleanup.

Optionally catch `GeneratorExit` only for cancellation-specific bookkeeping.

In [35]:
def safe_on_close(log: EventLog):
    try:
        yield "running"
    except GeneratorExit:
        log.add("cancel-request")
        raise
    finally:
        log.add("release-resource")

In [36]:
log = EventLog()
safe = safe_on_close(log)

assert next(safe) == "running"
result = safe.close()

assert result is None
assert_state(safe, "GEN_CLOSED")
assert log.names() == ["cancel-request", "release-resource"]

## Rule to remember

Inside `except GeneratorExit`:

- do not yield,
- perform only necessary cancellation work,
- then re-raise or return.

Keep unconditional cleanup in `finally`.

# Problem 6 — Cleanup order across a deep delegation chain

We will create four layers:

```text
application
  └── service
        └── repository
              └── connection
```

Each layer has a `finally` block.

The question is: in which order do they run when the outermost generator is closed?

## Step 1 — Define the four layers

In [37]:
def connection(log: EventLog):
    log.add("connection-open")
    try:
        while True:
            yield "connected"
    finally:
        log.add("connection-close")


def repository(log: EventLog):
    log.add("repository-enter")
    try:
        yield from connection(log)
    finally:
        log.add("repository-exit")


def service(log: EventLog):
    log.add("service-enter")
    try:
        yield from repository(log)
    finally:
        log.add("service-exit")


def application(log: EventLog):
    log.add("application-enter")
    try:
        yield from service(log)
    finally:
        log.add("application-exit")

## Step 2 — Prime the outermost generator

In [38]:
log = EventLog()
app = application(log)

assert next(app) == "connected"
log.show()

01. application-enter: None
02. service-enter: None
03. repository-enter: None
04. connection-open: None


All four layers have started.

Only the innermost generator is directly suspended at a `yield`. The other three are suspended at `yield from`.

## Step 3 — Close the application

Predict the cleanup order before running the next cell.

In [39]:
app.close()
log.show()

01. application-enter: None
02. service-enter: None
03. repository-enter: None
04. connection-open: None
05. connection-close: None
06. repository-exit: None
07. service-exit: None
08. application-exit: None


## Solution

The active innermost generator cleans up first.

Then each delegator unwinds outward:

```text
connection
repository
service
application
```

In [40]:
assert log.names() == [
    "application-enter",
    "service-enter",
    "repository-enter",
    "connection-open",
    "connection-close",
    "repository-exit",
    "service-exit",
    "application-exit",
]

This ordering is essential when outer resources depend on inner resources having already finished their cleanup.

# Problem 7 — Normal completion versus cancellation

A generator can produce a meaningful final result through `return`.

However, the caller must allow the generator to finish normally.

This problem compares:

- a sentinel-driven normal finish,
- explicit `throw(GeneratorExit)`,
- and `close()`.

## Step 1 — Define the processor

In [41]:
DONE = object()


def processor(log: EventLog):
    processed = 0

    try:
        item = yield "ready"

        while item is not DONE:
            processed += 1
            log.add("processed", item)
            item = yield processed

        return {"status": "complete", "processed": processed}

    except GeneratorExit:
        log.add("cancelled", processed)
        return {"status": "cancelled", "processed": processed}

    finally:
        log.add("cleanup")

## Experiment A — Normal completion with a sentinel

In [42]:
log_normal = EventLog()
normal = processor(log_normal)

assert next(normal) == "ready"
assert normal.send("a") == 1
assert normal.send("b") == 2

try:
    normal.send(DONE)
except StopIteration as exc:
    normal_result = exc.value
else:
    raise AssertionError("Expected normal completion")

print(normal_result)
log_normal.show()

{'status': 'complete', 'processed': 2}
01. processed: 'a'
02. processed: 'b'
03. cleanup: None


In [43]:
assert normal_result == {"status": "complete", "processed": 2}
assert log_normal.names() == ["processed", "processed", "cleanup"]

## Experiment B — Explicit `throw(GeneratorExit)`

Unlike `close()`, an explicit `throw()` exposes the terminating `StopIteration.value`.

In [44]:
log_throw = EventLog()
thrown = processor(log_throw)

assert next(thrown) == "ready"
assert thrown.send("x") == 1

try:
    thrown.throw(GeneratorExit())
except StopIteration as exc:
    thrown_result = exc.value
else:
    raise AssertionError("Expected StopIteration")

print(thrown_result)
log_throw.show()

{'status': 'cancelled', 'processed': 1}
01. processed: 'x'
02. cancelled: 1
03. cleanup: None


In [45]:
assert thrown_result == {"status": "cancelled", "processed": 1}

## Experiment C — `close()`

The result depends on the Python version.

In [46]:
log_close = EventLog()
closed = processor(log_close)

assert next(closed) == "ready"
assert closed.send("x") == 1

close_result = closed.close()

print("close() returned:", close_result)
print("Python 3.13+ behavior:", close_returns_values())

close() returned: {'status': 'cancelled', 'processed': 1}
Python 3.13+ behavior: True


In [47]:
if close_returns_values():
    assert close_result == {"status": "cancelled", "processed": 1}
else:
    assert close_result is None

assert_state(closed, "GEN_CLOSED")

## Design conclusion

For a successful final report, prefer normal completion:

```python
generator.send(DONE)
```

Use `close()` for cancellation and cleanup.

Do not make important business results depend only on version-sensitive close-time return behavior.

# Problem 8 — Delegating to a plain iterable

`yield from` can delegate to any iterable, not only a generator.

However, a plain list iterator does not implement the full coroutine protocol.

## Step 1 — Delegate to a list

In [48]:
def list_delegator():
    result = yield from [10, 20, 30]
    yield ("result", result)

## Step 2 — Consume normally

In [49]:
values, final_return = collect_with_return(list_delegator())

print(values)
print(final_return)

[10, 20, 30, ('result', None)]
None


In [50]:
assert values == [10, 20, 30, ("result", None)]
assert final_return is None

The list iterator terminates with an ordinary `StopIteration` whose value is `None`.

Therefore the `yield from` expression evaluates to `None`.

## Step 3 — Send a non-`None` value

A list iterator has no `send()` method.

Predict what happens when a caller sends `99` while delegation is active.

In [51]:
plain = list_delegator()
assert next(plain) == 10

try:
    plain.send(99)
except AttributeError as exc:
    send_error = exc
    print(type(exc).__name__, str(exc))
else:
    raise AssertionError("Expected AttributeError")

AttributeError 'list_iterator' object has no attribute 'send'


In [52]:
assert_state(plain, "GEN_CLOSED")

## Explanation

`yield from` supports the complete protocol only when the delegated object implements the relevant methods.

A list iterator supports:

- `__next__`

but not:

- `send`
- `throw`
- `close` as a generator protocol

## Step 4 — Wrap the iterable in a generator

A generator adapter can define how sent values should be treated.

In [53]:
def interactive_iterable(values, log: EventLog):
    for value in values:
        incoming = yield value
        log.add("caller-sent", incoming)

    return "adapter-finished"


def interactive_delegator(values, log: EventLog):
    result = yield from interactive_iterable(values, log)
    yield ("result", result)

In [54]:
log = EventLog()
interactive = interactive_delegator([10, 20, 30], log)

assert next(interactive) == 10
assert interactive.send("after-10") == 20
assert interactive.send("after-20") == 30
assert interactive.send("after-30") == ("result", "adapter-finished")
assert next(interactive, None) is None

log.show()

01. caller-sent: 'after-10'
02. caller-sent: 'after-20'
03. caller-sent: 'after-30'


The adapter turns a one-way iterable into a documented bidirectional protocol.

# Problem 9 — Recursive traversal with returned statistics

Recursive generators are a natural use case for `yield from`.

We will traverse a nested dictionary and:

- yield every leaf,
- return aggregate statistics from every recursive call,
- and combine those statistics at the parent level.

## Desired output

For this input:

```python
{
    "server": {
        "host": "localhost",
        "ports": {"http": 80, "https": 443},
    },
    "debug": False,
}
```

we want to yield path/value pairs such as:

```python
(("server", "host"), "localhost")
```

The final return value should include:

- number of leaves,
- number of dictionary branches,
- maximum depth.

## Step 1 — Define the recursive walker

In [55]:
def walk_with_stats(node: Any, path: tuple[str, ...] = ()):
    if not isinstance(node, dict):
        yield (path, node)
        return {
            "leaves": 1,
            "branches": 0,
            "max_depth": len(path),
        }

    total_leaves = 0
    total_branches = 1
    maximum_depth = len(path)

    for key, child in node.items():
        child_stats = yield from walk_with_stats(child, path + (key,))

        total_leaves += child_stats["leaves"]
        total_branches += child_stats["branches"]
        maximum_depth = max(maximum_depth, child_stats["max_depth"])

    return {
        "leaves": total_leaves,
        "branches": total_branches,
        "max_depth": maximum_depth,
    }

## Step 2 — Run the walker

Notice that yielded leaf data and returned aggregate metadata travel through different channels.

In [56]:
configuration = {
    "server": {
        "host": "localhost",
        "ports": {"http": 80, "https": 443},
    },
    "debug": False,
}

leaf_values, statistics = collect_with_return(walk_with_stats(configuration))

print("Leaves:")
for item in leaf_values:
    print(" ", item)

print("Statistics:", statistics)

Leaves:
  (('server', 'host'), 'localhost')
  (('server', 'ports', 'http'), 80)
  (('server', 'ports', 'https'), 443)
  (('debug',), False)
Statistics: {'leaves': 4, 'branches': 3, 'max_depth': 3}


In [57]:
assert leaf_values == [
    (("server", "host"), "localhost"),
    (("server", "ports", "http"), 80),
    (("server", "ports", "https"), 443),
    (("debug",), False),
]

assert statistics == {
    "leaves": 4,
    "branches": 3,
    "max_depth": 3,
}

## Why `yield from` is useful here

Each recursive call does two things:

1. yields an arbitrary number of leaves,
2. returns one statistics dictionary.

`yield from` forwards all yielded leaves and captures the single final statistics result.

# Problem 10 — A resource-owning source and an early consumer stop

A generator that owns a resource must release it during:

- normal exhaustion,
- exceptions,
- and early cancellation.

We will simulate a resource rather than open a real file.

## Step 1 — Define a resource object

In [58]:
@dataclass
class FakeResource:
    records: list[str]
    log: EventLog
    is_open: bool = False

    def open(self) -> None:
        self.is_open = True
        self.log.add("resource-open")

    def close(self) -> None:
        self.is_open = False
        self.log.add("resource-close")

## Step 2 — Define the resource-owning generator

In [59]:
def resource_source(resource: FakeResource):
    produced = 0
    resource.open()

    try:
        for record in resource.records:
            produced += 1
            yield record
        return {"produced": produced}
    finally:
        resource.close()

## Step 3 — Define a delegator

The delegator captures the source report and yields one final report.

In [60]:
def resource_session(resource: FakeResource):
    report = yield from resource_source(resource)
    yield ("source-report", report)

## Experiment A — Normal exhaustion

In [61]:
log_normal = EventLog()
resource_normal = FakeResource(["a", "b"], log_normal)

normal_values = list(resource_session(resource_normal))

print(normal_values)
log_normal.show()

['a', 'b', ('source-report', {'produced': 2})]
01. resource-open: None
02. resource-close: None


In [62]:
assert normal_values == ["a", "b", ("source-report", {"produced": 2})]
assert resource_normal.is_open is False
assert log_normal.names() == ["resource-open", "resource-close"]

## Experiment B — Early close

Consume one value and then close the outer session.

In [63]:
log_early = EventLog()
resource_early = FakeResource(["a", "b", "c"], log_early)
early_session = resource_session(resource_early)

assert next(early_session) == "a"
assert resource_early.is_open is True

early_session.close()

In [64]:
assert resource_early.is_open is False
assert_state(early_session, "GEN_CLOSED")
assert log_early.names() == ["resource-open", "resource-close"]

## Best practice

The generator that acquires a resource should usually own the corresponding cleanup in `finally`.

Delegation then preserves the cleanup behavior when the outer generator is closed.

# Problem 11 — Manually forwarding the protocol

A common misunderstanding is that this:

```python
for value in subgenerator:
    yield value
```

is equivalent to:

```python
yield from subgenerator
```

It is not equivalent for a bidirectional protocol.

We will build a simplified manual delegator to see why `yield from` is valuable.

## Supported subset

Our simplified implementation will support:

- `next()`
- `send()`
- `close()`
- capturing `StopIteration.value`

It intentionally omits complete `throw()` forwarding.

In [65]:
def simplified_yield_from(subiterator):
    iterator = iter(subiterator)

    try:
        current = next(iterator)
    except StopIteration as exc:
        return exc.value

    while True:
        try:
            sent = yield current
        except GeneratorExit:
            close_method = getattr(iterator, "close", None)

            if close_method is not None:
                close_method()

            raise
        else:
            try:
                if sent is None:
                    current = next(iterator)
                else:
                    send_method = getattr(iterator, "send")
                    current = send_method(sent)
            except StopIteration as exc:
                return exc.value

## Step 1 — Define a small dialogue generator

In [66]:
def dialogue(log: EventLog):
    name = yield "Name?"
    log.add("name", name)

    language = yield "Favorite language?"
    log.add("language", language)

    return {
        "name": name,
        "language": language,
    }

## Step 2 — Compare native and manual delegation

In [67]:
def native_dialogue(log: EventLog):
    result = yield from dialogue(log)
    yield ("complete", result)


def manual_dialogue(log: EventLog):
    result = yield from simplified_yield_from(dialogue(log))
    yield ("complete", result)

In [68]:
for factory in (native_dialogue, manual_dialogue):
    log = EventLog()
    conversation = factory(log)

    assert next(conversation) == "Name?"
    assert conversation.send("Ada") == "Favorite language?"
    assert conversation.send("Python") == (
        "complete",
        {"name": "Ada", "language": "Python"},
    )
    assert next(conversation, None) is None

    assert log.events == [
        ("name", "Ada"),
        ("language", "Python"),
    ]

## Why not write this manually in production?

The complete PEP 380 behavior also handles:

- exceptions thrown into the delegator,
- missing `throw()` or `close()` methods,
- traceback propagation,
- `GeneratorExit`,
- and several edge cases.

Native `yield from` is shorter, clearer, and more reliable.

# Problem 12 — Capstone: a transactional command session

We will combine the main ideas in one protocol.

The transaction generator will receive commands through `send()`.

Supported commands:

```python
("SET", key, value)
("DELETE", key)
COMMIT
```

Requirements:

- each successful operation yields an acknowledgment,
- invalid commands raise `ValueError`,
- `COMMIT` terminates normally and returns a transaction report,
- closing early records a rollback,
- a delegating session yields the final commit report.

## Step 1 — Define the commit sentinel

In [69]:
COMMIT = object()

## Step 2 — Define the transaction subgenerator

The transaction keeps staged data private until commit.

It also logs whether it committed or rolled back.

In [70]:
def transaction(log: EventLog):
    staged: dict[str, Any] = {}
    operation_count = 0
    committed = False

    log.add("transaction-begin")

    try:
        command = yield "transaction-ready"

        while command is not COMMIT:
            if (
                isinstance(command, tuple)
                and len(command) == 3
                and command[0] == "SET"
            ):
                _, key, value = command
                staged[key] = value
                operation_count += 1
                command = yield {
                    "ok": True,
                    "operation": "SET",
                    "key": key,
                }

            elif (
                isinstance(command, tuple)
                and len(command) == 2
                and command[0] == "DELETE"
            ):
                _, key = command
                staged.pop(key, None)
                operation_count += 1
                command = yield {
                    "ok": True,
                    "operation": "DELETE",
                    "key": key,
                }

            else:
                raise ValueError(f"invalid command: {command!r}")

        committed = True
        log.add("transaction-commit", dict(staged))

        return {
            "status": "committed",
            "operations": operation_count,
            "data": dict(staged),
        }

    finally:
        if not committed:
            log.add("transaction-rollback", dict(staged))

        log.add("transaction-end")

## Step 3 — Define the delegating session

In [71]:
def transaction_session(log: EventLog):
    log.add("session-begin")

    try:
        report = yield from transaction(log)
        yield ("commit-report", report)
    finally:
        log.add("session-end")

## Experiment A — Commit normally

Run the protocol one step at a time.

In [72]:
log_commit = EventLog()
session_commit = transaction_session(log_commit)

assert next(session_commit) == "transaction-ready"

assert session_commit.send(("SET", "language", "Python")) == {
    "ok": True,
    "operation": "SET",
    "key": "language",
}

assert session_commit.send(("SET", "version", 3)) == {
    "ok": True,
    "operation": "SET",
    "key": "version",
}

assert session_commit.send(("DELETE", "version")) == {
    "ok": True,
    "operation": "DELETE",
    "key": "version",
}

Sending `COMMIT` causes the transaction to return.

The session captures that return value and yields a commit report.

In [73]:
commit_output = session_commit.send(COMMIT)

print(commit_output)
log_commit.show()

('commit-report', {'status': 'committed', 'operations': 3, 'data': {'language': 'Python'}})
01. session-begin: None
02. transaction-begin: None
03. transaction-commit: {'language': 'Python'}
04. transaction-end: None


In [74]:
assert commit_output == (
    "commit-report",
    {
        "status": "committed",
        "operations": 3,
        "data": {"language": "Python"},
    },
)

assert next(session_commit, None) is None

assert log_commit.names() == [
    "session-begin",
    "transaction-begin",
    "transaction-commit",
    "transaction-end",
    "session-end",
]

## Experiment B — Cancel and roll back

Create another transaction, stage data, and close the outer session before commit.

In [75]:
log_rollback = EventLog()
session_rollback = transaction_session(log_rollback)

assert next(session_rollback) == "transaction-ready"
assert session_rollback.send(("SET", "temporary", 99))["ok"] is True

session_rollback.close()

In [76]:
assert_state(session_rollback, "GEN_CLOSED")

assert log_rollback.events == [
    ("session-begin", None),
    ("transaction-begin", None),
    ("transaction-rollback", {"temporary": 99}),
    ("transaction-end", None),
    ("session-end", None),
]

log_rollback.show()

01. session-begin: None
02. transaction-begin: None
03. transaction-rollback: {'temporary': 99}
04. transaction-end: None
05. session-end: None


## Capstone explanation

This example uses two distinct completion paths.

### Commit

```text
caller sends COMMIT
transaction returns report
yield from captures report
session yields final report
```

### Cancellation

```text
caller closes session
close propagates to transaction
transaction finally records rollback
session finally records session cleanup
```

This is a useful general design:

- normal messages represent successful workflow decisions,
- `return` carries final metadata,
- `yield from` propagates the protocol,
- and `close()` represents cancellation.

# Additional guided mini-problems

These shorter problems provide more lines of code and more opportunities to experiment.

## Mini-problem A — An empty subgenerator with a return value

Predict the first value produced by `next(wrapper())`.

In [77]:
def empty_subgenerator():
    if False:
        yield None
    return 500


def empty_wrapper():
    result = yield from empty_subgenerator()
    yield result


empty = empty_wrapper()
assert next(empty) == 500
assert next(empty, None) is None

Because the subgenerator yields nothing, `yield from` completes immediately with the returned value `500`.

## Mini-problem B — Capturing locals before and after completion

In [78]:
def local_state():
    total = 10
    incoming = yield total
    total += incoming
    yield total
    return total


stateful = local_state()

assert getgeneratorlocals(stateful) == {}
assert next(stateful) == 10

created_locals = getgeneratorlocals(stateful)
print(created_locals)

assert created_locals["total"] == 10
assert stateful.send(5) == 15

try:
    next(stateful)
except StopIteration as exc:
    assert exc.value == 15

assert getgeneratorlocals(stateful) == {}

{'total': 10}


Closed generator frames no longer expose the suspended local-variable mapping.

## Mini-problem C — `send(None)` and `next()` are equivalent during delegation

In [79]:
def simple_values():
    yield "first"
    yield "second"


def simple_parent():
    yield from simple_values()


simple = simple_parent()

assert next(simple) == "first"
assert simple.send(None) == "second"
assert next(simple, None) is None

For an active generator:

```python
next(g)
```

and:

```python
g.send(None)
```

both request the next yielded value.

## Mini-problem D — A subgenerator that returns immediately after receiving data

In [80]:
def one_message():
    message = yield "waiting"
    return message.upper()


def one_message_parent():
    result = yield from one_message()
    yield ("upper", result)


single = one_message_parent()

assert next(single) == "waiting"
assert single.send("hello") == ("upper", "HELLO")
assert next(single, None) is None

The sent value becomes the result of the suspended `yield` expression inside the subgenerator.

# Summary table

| Event at the caller | Behavior during active `yield from` |
|---|---|
| `next(delegator)` | Requests the next item from the subiterator |
| `delegator.send(None)` | Also requests the next item |
| `delegator.send(value)` | Calls the subiterator's `send(value)` |
| `delegator.throw(exc)` | Routes the exception to the subiterator when supported |
| `delegator.close()` | Requests subiterator closure, then closes outward |
| subgenerator `yield value` | Value travels to the original caller |
| subgenerator `return result` | `yield from` evaluates to `result` |
| child closed directly | Parent remains suspended until resumed |

# Best-practice checklist

Before using generators as a protocol, ask:

1. What values are yielded outward?
2. What values may be sent inward?
3. What message means successful completion?
4. What does cancellation mean?
5. Which exceptions are recoverable?
6. Which generator owns each resource?
7. What final metadata is returned?
8. How will the protocol be tested?
9. Does any delegated object lack `send()`, `throw()`, or `close()`?
10. Does cleanup remain correct under early termination?

# Further challenge problems

## Challenge 1 — Nested transaction

Create an outer transaction that delegates to several child transactions. The outer transaction should commit only if all child reports indicate success.

## Challenge 2 — Exception retry budget

Build a worker that recovers from at most three injected `ValueError` exceptions. On the fourth exception, return a failure report.

## Challenge 3 — Recursive cancellation trace

Create a recursive generator chain of configurable depth. Close the root and verify exact cleanup order for depths 1, 5, and 20.

## Challenge 4 — Message batching

Receive values through `send()`. Yield a completed batch every five values. On a sentinel, return a partial-batch report.

## Challenge 5 — Generator adapter

Wrap a plain iterable so that sent commands can skip, repeat, or replace the next value.

## Challenge 6 — Context-managed session

Write a context manager that primes a generator on entry and always closes it on exit.

## Challenge 7 — Protocol transcript

Write a driver that records every caller operation and every yielded or returned value, producing a human-readable protocol transcript.

# Final notebook verification

The final cell performs a small end-to-end check using the transaction capstone.

In [81]:
verification_log = EventLog()
verification = transaction_session(verification_log)

assert next(verification) == "transaction-ready"
assert verification.send(("SET", "verified", True))["ok"] is True

final_item = verification.send(COMMIT)

assert final_item[0] == "commit-report"
assert final_item[1]["data"] == {"verified": True}
assert next(verification, None) is None

print("All tutorial solution cells executed successfully.")

All tutorial solution cells executed successfully.
